<a href="https://colab.research.google.com/github/ebolofis/Data-Science-Machine-Learning/blob/main/EDA_Feature_selection_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libraries

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
import os
from pathlib import Path
import requests
from io import BytesIO

# --- Advanced Analysis Imports ---
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [12]:
# Set plot style and ignore warnings for cleaner output
sns.set_style("whitegrid")
warnings.filterwarnings("ignore")

In [14]:
from google.colab import drive
drive.mount('/content/drive')

os.chdir('/content/drive/My Drive/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Define utility functions

## Functions related to data loading and quality assessment

In [16]:
def load_data_from_git_hub(url_sales: str, url_macro: str, github_token_file: str) -> pd.DataFrame:
  try:
    with open(github_token_file, 'r') as f:
        github_token = f.read().strip()

    # Define the request headers with the PAT for authentication
    headers = {
        'Authorization': f'token {github_token}',
        'Accept': 'application/vnd.github.v3.raw'
        }

    # Fetch the file content
    print(f"Attempting to fetch file from: {url_sales} and {url_macro}")
    response_1 = requests.get(url_sales, headers=headers)
    response_2 = requests.get(url_macro, headers=headers)

    if response_1.status_code == 200:
      file_content_1 = BytesIO(response_1.content)
      df_sales = pd.read_excel(file_content_1)
      print("✅ Sales file fetched successfully from GitHub.")
    else:
      print("❌ Error fetching sales file. ")
      raise Exception(f"Status Code: {response_1.status_code} Message: {response_1.text}")

    if response_2.status_code == 200:
      file_content_2 = BytesIO(response_2.content)
      df_macro = pd.read_excel(file_content_2)
      print("✅ Exogenous file fetched successfully from GitHub.")
    else:
      print("❌ Error fetching exogenous file. ")
      raise Exception(f"Status Code: {response_2.status_code} Message: {response_1.text}")

  except Exception as e:
    print("❌ Error occured while fetching file from Github")
    print(e)
    raise

  return df_sales, df_macro


In [17]:
def load_and_merge_data(sales_path: str, exog_path: str, from_git_hub = False, git_hub_token_file= None) -> pd.DataFrame:
    """Load sales + exogenous Excel files and merge on Month."""

    if from_git_hub:
      sales_df, exo_df = load_data_from_git_hub(sales_path, exog_path, git_hub_token_file)
    else:
      sales_df = pd.read_excel(sales_path)
      exo_df = pd.read_excel(exog_path)


    sales_df["Month"] = pd.to_datetime(sales_df["Month"], errors='coerce')
    exo_df["Month"] = pd.to_datetime(exo_df["Month"], errors='coerce')

    macro_var = [c for c in exo_df.columns if c != "Month"]

    sales_df = sales_df.sort_values("Month")
    exo_df = exo_df.sort_values("Month")

    df_merged = pd.merge(sales_df, exo_df, on="Month", how="inner")
    df_merged = df_merged.dropna().reset_index(drop=True)
    #df_merged = df_merged.set_index("Month")

    # Create time features for analysis
    # df_merged['Year'] = df_merged.index.year
    # df_merged['Month_Num'] = df_merged.index.month

    # if "Cars" in df_merged.columns:
    #     df_merged = df_merged.rename(columns={"Cars": "Car_Sales"})

    #data_quality_check(df_merged)

    return df_merged, macro_var

In [18]:
def data_quality_check(df_merged: pd.DataFrame) -> None:
  # ==========================================
  # DATA HYGIENE CHECK
  # ==========================================

  # 1. CHECK FOR MISSING VALUES (NaNs)
  print("--- 1. Missing Values per Column ---")
  missing_vals = df_merged.isnull().sum()
  print(missing_vals[missing_vals > 0])
  if missing_vals.sum() == 0:
      print("✅ No missing values found in the merged dataset.")

  # 2. CHECK FOR DUPLICATES
  print("\n--- 2. Duplicate Checks ---")
  # Check for exact row duplicates
  duplicates = df_merged.duplicated().sum()
  # Check for duplicate DATES (Critical for Time Series)
  date_duplicates = df_merged['Month'].duplicated().sum()

  print(f"Duplicate Rows: {duplicates}")
  print(f"Duplicate Months: {date_duplicates}")

  if date_duplicates > 0:
      print("❌ CRITICAL: You have multiple entries for the same month. Aggregation is required.")
  else:
      print("✅ Time index is unique.")

  # 3. CHECK FOR "HIDDEN" GAPS (Discontinuous Time Series)
  # A time series must have a row for every month.
  # If Jan and Mar exist but Feb is missing, models will fail.
  print("\n--- 3. Time Continuity Check ---")
  min_date = df_merged['Month'].min()
  max_date = df_merged['Month'].max()

  # Generate a perfect monthly timeline
  expected_dates = pd.date_range(start=min_date, end=max_date, freq='MS') # MS = Month Start

  if len(df_merged) == len(expected_dates):
      print(f"✅ Timeline is perfect. {len(df_merged)} months present from {min_date.date()} to {max_date.date()}.")
  else:
      print(f"❌ GAP DETECTED!")
      print(f"Expected {len(expected_dates)} months, but found {len(df_merged)}.")
      # Find which dates are missing
      missing_dates = expected_dates.difference(df_merged['Month'])
      print(f"Missing Dates: {missing_dates}")

## Functions related to EDA

In [19]:
def plot_sale_hist_trend (df_merged, sale_col ):
 # --- PLOT 1: History, Trend & Volatility ---
  plt.figure(figsize=(14, 7))

  # Calculate Rolling Mean (Trend) and Std Dev (Volatility) over 12 months
  rolling_mean = df_merged[sale_col].rolling(window=12).mean()
  rolling_std = df_merged[sale_col].rolling(window=12).std()

  # Plot
  plt.plot(df_merged.index, df_merged[sale_col], label='Total Sales (Actual)', color='#003366', alpha=0.6)
  plt.plot(rolling_mean, label='Trend (12-Month Moving Avg)', color='orange', linewidth=2.5)

  # Volatility Band (Bollinger Band style)
  plt.fill_between(df_merged.index,
                    rolling_mean - rolling_std,
                    rolling_mean + rolling_std,
                    color='orange', alpha=0.1, label='Volatility (±1 Std Dev)')

  # COVID Zone
  plt.axvspan(pd.to_datetime('2020-03-01'), pd.to_datetime('2021-01-01'), color='red', alpha=0.1, label='COVID Shock')

  plt.title('Historical Analysis: Sales, Trend, and Volatility', fontsize=16)
  plt.legend(loc='upper left')
  plt.grid(True, alpha=0.3)
  plt.show()


In [20]:
def plot_dist_seasonality(df_merged, sale_col):
  # ==========================================
  # DISTRIBUTION & SEASONALITY
  # ==========================================

  # --- Sales Distribution ---
  plt.figure(figsize=(14, 6))

  plt.subplot(1, 2, 1)
  sns.histplot(df_merged[sale_col], kde=True, color='#003366')
  plt.title('Sales Volume Distribution', fontsize=14)
  plt.xlabel('Units Sold')

  # QQ Plot (Normality Check)
  plt.subplot(1, 2, 2)
  sm.qqplot(df_merged[sale_col], line='s', ax=plt.gca())
  plt.title('Q-Q Plot (Normality Test)', fontsize=14)
  plt.tight_layout()
  plt.show()

  # --- Monthly Seasonality (Boxplot) ---
  plt.figure(figsize=(12, 6))
  # Exclude 2020 to analyze "Normal" seasonality
  df_season = df_merged[~df_merged['Year'].isin([2020])]

  sns.boxplot(data=df_season, x='Month_Num', y=sale_col, palette="Blues")
  plt.title('Monthly Seasonality (Excluding COVID)', fontsize=16)
  plt.xlabel('Month')
  plt.xticks(ticks=range(0, 12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
  plt.grid(True, axis='y', alpha=0.3)
  plt.show()

In [21]:
def corr_EDA_analysis (df_merged, sale_col):
   # ==========================================
    # CORRELATIONS (DRIVERS)
    # ==========================================
    print("\n" + "="*50)
    print("3. CORRELATION ANALYSIS")
    print("="*50)

    # Select numeric columns only
    numeric_cols = df_merged.select_dtypes(include=['float64', 'int64']).columns
    # Exclude technical columns
    corr_cols = [c for c in numeric_cols if c not in ['Year', 'Month_Num']]

    corr_matrix = df_merged[corr_cols].corr()

    # --- PLOT 5: Heatmap ---
    plt.figure(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    sns.heatmap(corr_matrix, mask=mask, cmap='RdBu', center=0, annot=False, vmin=-1, vmax=1)
    plt.title('Global Correlation Matrix', fontsize=16)
    plt.show()

    # --- Top Drivers ---
    print("\n--- Top 5 Positive Correlations with Sales (Cars) ---")
    print(corr_matrix[sale_col].sort_values(ascending=False).head(6))
    print("\n--- Top 3 Negative Correlations (Headwinds) ---")
    print(corr_matrix[sale_col].sort_values(ascending=True).head(3))

    # ==========================================
    # SCATTER PLOTS (KEY RELATIONSHIPS)
    # ==========================================
    top_drivers = corr_matrix[sale_col].abs().sort_values(ascending=False).index[1:4] # Exclude 'Cars'

    plt.figure(figsize=(15, 5))
    for i, col in enumerate(top_drivers):
        plt.subplot(1, 3, i+1)
        sns.regplot(data=df_merged, x=col, y=sale_col,
                    scatter_kws={'alpha':0.5, 'color':'#003366'},
                    line_kws={'color':'red'})
        plt.title(f'Sales vs {col}\n(Corr: {corr_matrix.loc[sale_col, col]:.2f})')

    plt.tight_layout()
    plt.show()

## Functions related to time-series stationary analysis

In [22]:
def perform_adf_test(series, title=''):
    """Performs Augmented Dickey-Fuller Test for Stationarity"""
    print(f'\nAugmented Dickey-Fuller Test: {title}')
    result = adfuller(series.dropna(), autolag='AIC')
    labels = ['ADF Test Statistic', 'p-value', '# Lags Used', 'Number of Observations']
    out = pd.Series(result[0:4], index=labels)

    for key, val in result[4].items():
        out[f'Critical Value ({key})'] = val

    print(out)
    if result[1] <= 0.05:
        print("✅ Conclusion: Strong evidence against Null Hypothesis. Series is STATIONARY.")
        return True
    else:
        print("❌ Conclusion: Weak evidence against Null Hypothesis. Series is NON-STATIONARY.")
        return False


def perform_ts_stationary_analysis(df_merged, sale_col):

# --- Stationarity Analysis (ADF & ACF/PACF) ---
  print("\n Stationarity Analysis ---")

  # Check Original Series
  ts = df_merged[sale_col].asfreq('MS').ffill()
  is_stationary = perform_adf_test(ts, "Original Series")

  # Plot ACF/PACF for Original
  fig, axes = plt.subplots(1, 2, figsize=(16, 5))
  plot_acf(ts.dropna(), lags=36, ax=axes[0], title='ACF - Original Series')
  plot_pacf(ts.dropna(), lags=36, ax=axes[1], title='PACF - Original Series')
  plt.show()

  # If not stationary, check First Difference
  if not is_stationary:
      print("\n--- Testing First Difference (d=1) ---")
      ts_diff = ts.diff().dropna()
      is_stationary_diff = perform_adf_test(ts_diff, "First Difference")

      fig, axes = plt.subplots(1, 2, figsize=(16, 5))
      plot_acf(ts_diff, lags=36, ax=axes[0], title='ACF - First Difference (d=1)')
      plot_pacf(ts_diff, lags=36, ax=axes[1], title='PACF - First Difference (d=1)')
      plt.show()

## Function related to key driver selection

In [23]:
# =========================
# LAGGED CORRELATION ANALYSIS
# =========================

def compute_lagged_correlations(df: pd.DataFrame,
                                target_col:str,
                                macro_vars: list,
                                max_lag: int = 12) -> pd.DataFrame:
    """
    Compute correlation between Car_Sales(t) and X(t - lag)
    for each macro variable and lag = 0..max_lag.
    """
    records = []
    y = df[target_col]

    for var in macro_vars:
        x_full = df[var]
        for lag in range(0, max_lag + 1):
            x = x_full.shift(lag)
            valid = x.notna() & y.notna()
            if valid.sum() > 5:
                corr = y[valid].corr(x[valid])
            else:
                corr = np.nan
            records.append({
                "Variable": var,
                "Lag_Months": lag,
                "Correlation": corr
            })

    lag_corr_df = pd.DataFrame(records)
    lag_corr_df["AbsCorr"] = lag_corr_df["Correlation"].abs()

    # Best lag per variable
    best_corr_df = (lag_corr_df
                    .sort_values(["Variable", "AbsCorr"], ascending=[True, False])
                    .groupby("Variable")
                    .head(1)
                    .reset_index(drop=True))

    return lag_corr_df, best_corr_df


# =========================
# GRANGER CAUSALITY
# =========================

def granger_analysis(df: pd.DataFrame,
                     target_col:str,
                     macro_vars: list,
                     max_lag: int = 12) -> pd.DataFrame:
    """
    Run Granger causality tests:
    Does macro variable X help predict Car_Sales?

    statsmodels expects data of shape [y, x] for grangercausalitytests.
    """
    results = []

    for var in macro_vars:
        try:
            sub = df[[target_col, var]].dropna()
            if len(sub) < max_lag + 5:
                # Not enough data for a meaningful Granger test
                results.append({
                    "Variable": var,
                    "Best_Lag": None,
                    "Best_p_value": np.nan
                })
                continue

            data = sub[[target_col, var]].to_numpy()
            res = grangercausalitytests(data, maxlag=max_lag, verbose=False)

            best_p = np.inf
            best_lag = None
            for lag, stats_dict in res.items():
                # 'ssr_ftest' is a common choice; you can inspect others
                pval = stats_dict[0]['ssr_ftest'][1]
                if pval < best_p:
                    best_p = pval
                    best_lag = lag

            results.append({
                "Variable": var,
                "Best_Lag": best_lag,
                "Best_p_value": best_p
            })

        except Exception as e:
            results.append({
                "Variable": var,
                "Best_Lag": None,
                "Best_p_value": np.nan
            })

    granger_df = pd.DataFrame(results)
    granger_df = granger_df.sort_values("Best_p_value")

    return granger_df


In [24]:
# =========================
# LASSO WITH LAGGED FEATURES
# =========================
def build_lagged_feature_matrix(df: pd.DataFrame,
                                tartget_col: str,
                                macro_vars: list,
                                max_lag: int) -> pd.DataFrame:
    """
    Build a dataset with:
    - Lagged Car_Sales (1..max_lag)
    - Lagged macro vars (1..max_lag)
    - Target: Car_Sales shifted -horizon
    """
    work_df = df.copy().sort_index()

    vars_to_be_lagged = macro_vars +[tartget_col]
    lagged_features_col = []
    # Create lag features
    for var in vars_to_be_lagged:
      for lag in range(1, max_lag + 1):
        work_df[f"{var}_lag{lag}"] = work_df[var].shift(lag)
        lagged_features_col.append(f"{var}_lag{lag}")


    # for lag in range(1, max_lag + 1):
    #     # work_df[f"Car_Sales_lag{lag}"] = work_df[tartget_col].shift(lag)
    #     for var in vars_to_be_lagged:
    #         work_df[f"{var}_lag{lag}"] = work_df[var].shift(lag)
    #         lagged_features_col.append(f"{var}_lag{lag}")

    # work_df[f"Target_{horizon}m"] = work_df[tartget_col].shift(-horizon)
    # lagged_features_col.append(f"Target_{horizon}m")

    # Drop NA from lags and target
    work_df = work_df.dropna()#.reset_index(drop=True)
    return work_df, lagged_features_col

def parse_feature_name(fname: str):
    # pattern: <var>_lag<k>
    if "_lag" in fname:
        var, lag_part = fname.rsplit("_lag", 1)
        try:
            lag = int(lag_part)
        except ValueError:
            lag = 0
        return var, lag
    return fname, None




In [25]:
def lasso_lag_selection(model_df: pd.DataFrame,
                        max_lag: int,
                        target_col: str,
                        lagged_feature_cols: list,
                        horizon: int = 1) -> pd.DataFrame:
    """
    Run LassoCV to select relevant lagged features and lags.

    Returns:
        lasso_summary: DataFrame with Base_Variable, Lag_Months, Total_Abs_Coeff
    """
    # Collect only lag features as X
    # feature_cols = [
    #     c for c in model_df.columns
    #     if "_lag" in c and any(c.endswith(f"_lag{lag}") for lag in range(1, max_lag + 1))
    # ]

    X = model_df[lagged_feature_cols].values
    y = model_df[target_col].values


    lasso = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LassoCV(cv=5, random_state=42, max_iter=10000))
    ])

    lasso.fit(X, y)

    alpha = lasso.named_steps["model"].alpha_
    coef = lasso.named_steps["model"].coef_

    lasso_results = pd.DataFrame({
        "Feature": feature_cols,
        "Coefficient": coef
    })

    # Filter non-zero (or near non-zero) coefficients
    lasso_selected = lasso_results[lasso_results["Coefficient"].abs() > 1e-5].copy()
    lasso_selected = lasso_selected.sort_values(
        "Coefficient", key=lambda s: s.abs(), ascending=False
    )



    lasso_selected["Base_Variable"], lasso_selected["Lag_Months"] = zip(
        *lasso_selected["Feature"].map(parse_feature_name)
    )

    lasso_summary = (
        lasso_selected
        .groupby(["Base_Variable", "Lag_Months"])
        .agg(Total_Abs_Coeff=("Coefficient", lambda x: x.abs().sum()))
        .reset_index()
        .sort_values("Total_Abs_Coeff", ascending=False)
    )

    print(f"\n[INFO] LassoCV selected alpha = {alpha:.6f}")
    return lasso_summary


# Data preprocessing and EDA

##Loading data and assessment on data qualities

In [26]:
SALE_DATA_PATH = "Vehicle_Data_Chile_2010_2025.xlsx"
EXOG_DATA_PATH = "Exogenous_Variables_Chile.xlsx"

In [27]:
merged_data_df, macro_var = load_and_merge_data(SALE_DATA_PATH, EXOG_DATA_PATH)
merged_data_df.info()
print(macro_var)
data_quality_check(merged_data_df)

FileNotFoundError: [Errno 2] No such file or directory: 'Vehicle_Data_Peru_2010_2025.xlsx'

In [ ]:
data_quality_check(merged_data_df)

In [ ]:
# Set Month as Index for time series analysis
merged_data_df.set_index('Month', inplace=True)
merged_data_df.sort_index(inplace=True)

print(f"Merged Data Shape: {merged_data_df.shape}")

# Create time features for analysis
merged_data_df['Year'] = merged_data_df.index.year
merged_data_df['Month_Num'] = merged_data_df.index.month

In [ ]:
print("\n--- Descriptive Statistics (Snapshot) ---")
print(merged_data_df.describe())

##EDA analysis

### EAD on trend/volitility and distributions of car sale volume

In [ ]:
plot_sale_hist_trend(merged_data_df, sale_col="Cars")

In [ ]:
plot_dist_seasonality(merged_data_df, "Cars")

###Correlation heatmap

In [ ]:
corr_EDA_analysis(merged_data_df, "Cars")

### Time series stationary analysis

In [ ]:
perform_ts_stationary_analysis(merged_data_df, "Cars")

# Key drivers identification

## Analysis on correlation among car sales and macro variables

In [ ]:
MAX_LAG = 12

In [ ]:

# 2. Lagged correlations
print("\n Computing correlations between target dependent variable and lagged independent variables: ")
lag_corr_df, best_corr_df = compute_lagged_correlations(
    df=merged_data_df,
    target_col= "Cars",
    macro_vars = macro_var,
    max_lag=MAX_LAG
)

print("Best lag per variable:")
print(best_corr_df.sort_values("Correlation", ascending=False))

In [ ]:
print("\nTop 20 variables by best absolute correlation (any lag):")
print(
    lag_corr_df.sort_values("AbsCorr", ascending=False)
    .head(20)
    .to_string(index=False)
)

## Conducting granger causality analysis

In [ ]:
granger_res_df = granger_analysis(df = merged_data_df,
                                  target_col = "Cars",
                                  macro_vars= macro_var,
                                  max_lag= 12)

In [ ]:
print("-"*60)
print("GRANGER CAUSALITY ANALYSIS")
print(" ","Showing the most predictive lag for each macro variable")
print("-"*60)

granger_res_df

## Lasso feature selection process

In [ ]:

lagged_feature_df, lagged_feature_cols = build_lagged_feature_matrix(df = merged_data_df,
                                                tartget_col= "Cars",
                                                macro_vars= macro_var,
                                                max_lag= MAX_LAG)

In [ ]:
lasso_lag_selection_res = lasso_lag_selection(model_df=lagged_feature_df,
                                          max_lag= MAX_LAG,
                                          target_col= "Cars",
                                          lagged_feature_cols= lagged_feature_cols)


In [ ]:
print("-"*60)
print("LASSO LAGGED VARIABLE SELECTION")
print(" ","Showing the most significant macro variable")
print("-"*60)

lasso_lag_selection_res

## save outputs to CSV (Optional)

In [ ]:
# 5. Optional: save outputs to Excel/CSV
print("\n Saving outputs to CSV...")
lag_corr_df.to_csv("lagged_correlations_full.csv", index=False)
best_corr_df.to_csv("lagged_correlations_best_by_var.csv", index=False)
granger_res_df.to_csv("granger_results.csv", index=False)
lasso_lag_selection_res.to_csv("lasso_selected_lags.csv", index=False)

print("\n[DONE] Analysis complete. CSV files saved in current directory:")
print(" - lagged_correlations_full.csv")
print(" - lagged_correlations_best_by_var.csv")
print(" - granger_results.csv")
print(" - lasso_selected_lags.csv")

# Time-series modelling using Sarimax

#Time-series modelling using XGBoost